In [0]:
p#%%
from pyspark.sql.types import IntegerType, StructType, StructField, StringType
from pyspark.sql.functions import col, initcap, trim
from datetime import datetime
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("clean_data").getOrCreate()

# -------------------------
# Create widgets
# -------------------------

dbutils.widgets.text("input_path","")
dbutils.widgets.text("output_path","")

input_path = dbutils.widgets.get("input_path")
output_path = dbutils.widgets.get("output_path")

# -------------------------
# Define schema
# -------------------------
input_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True)
])

# -------------------------
# Read CSV file
# -------------------------
df = (spark.read
        .option("header", True)
        .schema(input_schema)
        .csv(input_path)
     )

# -------------------------
# Data Cleaning
# -------------------------

# Remove duplicate IDs
df = df.dropDuplicates(["id"])

# Remove null IDs
df = df.filter(col("id").isNotNull())

# Change name first letter to uppercase
df = df.withColumn("name", initcap(trim(col("name"))))

# Remove invalid email IDs
df = df.filter(col("email").rlike(r"^[a-zA-Z0-9+._-]+@[a-zA-Z0-9+.-]+\.[a-zA-Z]{2,}$"))

# Remove invalid phone numbers
df = df.filter(col("phone").rlike(r"^[0-9]{10}$"))

df.show(truncate=False)

# -------------------------
# Add dynamic date
# -------------------------
date = datetime.now().strftime("%Y_%m_%d")

file_name = f"{output_path}/formatted_userdata_{date}"
print(file_name)

# -------------------------
# Write output
# -------------------------
df.write.mode("overwrite").parquet(file_name)